# Isoglot 03: does Tiny Aya recognise harm in Hausa, Yoruba and Igbo?

The baseline showed that the refusals Tiny Aya loses in these languages mostly become
confused (**Deflected**) answers, not harmful ones. This notebook asks why, which decides the repair:

- **Harm recognised, refusal does not fire:** the harmful prompts it deflected sit on the
  refusal direction like the prompts it refused → conditional refusal steering (SafeTune).
- **Harm not recognised:** deflected harmful prompts sit with benign prompts → data repair
  (CuratorKIT + fine-tuning), because steering would also hit harmless requests.

The English refusal direction is built from **BeaverTails** harmful prompts vs **Alpaca**
instructions (difference of means at the last prompt token, per layer, via SafeTune's
activation extractor). TukaBench prompts are only projected onto it, never used to build it.
No paid API calls.

**Before running:** Settings → Accelerator **GPU T4 x2**, Internet **on**; Secret **`HF_TOKEN`**
ticked. Then **Save Version → Save & Run All (Commit)**.


In [ ]:
%%capture
!pip install -q "isoglot[gpu,mechanism] @ git+https://github.com/nftkingiii/isoglot"


In [ ]:
import os, json, time, torch
from pathlib import Path
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

from huggingface_hub import HfApi, hf_hub_download, whoami
api = HfApi()
REPO = f"{whoami()['name']}/isoglot-runs"
RUNS = Path("/kaggle/working/runs")
for f in ["base/answers.jsonl", "base/judged.jsonl"]:
    hf_hub_download(REPO, f, repo_type="dataset", local_dir=RUNS)

prompt_of = {r["id"]: r["prompt"] for r in map(json.loads, open(RUNS / "base/answers.jsonl", encoding="utf-8"))}
judged = [r for r in map(json.loads, open(RUNS / "base/judged.jsonl", encoding="utf-8"))
          if r["label"] in ("Jailbroken", "Refused", "Deflected")]
for r in judged:
    r["prompt"] = prompt_of[r["id"]]
print(len(judged), "judged rows with prompts")


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
MODEL = "CohereLabs/tiny-aya-global"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16, device_map="cuda")
model.eval()
from safetune._refusal_helpers import _get_decoder_layers
N_LAYERS = len(_get_decoder_layers(model))
print(MODEL, "|", N_LAYERS, "layers |", torch.cuda.get_device_name(0))


## English refusal direction (outside the evaluation set)

In [ ]:
from isoglot.mechanism import direction_prompts, last_token_acts, mean_diff_directions

exclude = {r["goal_en"].lower() for r in judged}
harmful_en, harmless_en = direction_prompts(n=128, seed=0, exclude=exclude)
t0 = time.time()
directions = mean_diff_directions(last_token_acts(model, tok, harmful_en),
                                  last_token_acts(model, tok, harmless_en))
print(f"directions for {len(directions)} layers in {time.time()-t0:.0f}s")


## Project each language's TukaBench prompts onto it

In [ ]:
from isoglot import LANGS
from isoglot.mechanism import language_table, project

tables = {}
for lang in LANGS:
    rows = [r for r in judged if r["lang"] == lang]
    t0 = time.time()
    proj = project(last_token_acts(model, tok, [r["prompt"] for r in rows]), directions)
    tables[lang] = language_table(proj, rows)
    print(f"{lang}: {len(rows)} prompts in {time.time()-t0:.0f}s")


## Verdict

TukaBench's benign prompts are the benign twins of the harmful ones (same topics, e.g. a story about drug use), so harmful-vs-benign separation is a hard test; English sets the bar.

In [ ]:
from isoglot.mechanism import RECOGNISED, pick_layer, verdict

LAYER = pick_layer(tables["eng"], N_LAYERS)
eng_sep = next(x for x in tables["eng"] if x["layer"] == LAYER)["harm_vs_benign"]
if eng_sep < RECOGNISED:
    print(f"WARNING: the direction separates English harmful/benign prompts only at AUROC {eng_sep:.2f}; "
          "the verdicts below are not reliable.")
print(f"layer {LAYER} of {N_LAYERS} (best English harm/benign separation, first 20% skipped); "
      f"recognised threshold {RECOGNISED}")
print(f"{'lang':5}{'harm_vs_benign':>16}{'deflected_vs_benign':>21}{'refused_vs_deflected':>22}"
      f"{'n_refused':>11}{'n_deflected':>13}   verdict")
for lang in LANGS:
    t = next(x for x in tables[lang] if x["layer"] == LAYER)
    print(f"{lang:5}{t['harm_vs_benign']:16.3f}{t['deflected_vs_benign']:21.3f}{t['refused_vs_deflected']:22.3f}"
          f"{t['n_refused']:11d}{t['n_deflected']:13d}   {verdict(t)}")


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for lang in LANGS:
    layers = [t["layer"] for t in tables[lang]]
    axes[0].plot(layers, [t["harm_vs_benign"] for t in tables[lang]], label=lang)
    axes[1].plot(layers, [t["deflected_vs_benign"] for t in tables[lang]], label=lang)
for ax, title in zip(axes, ["all harmful vs benign", "deflected harmful vs benign"]):
    ax.axhline(0.5, color="grey", lw=0.8, ls=":"); ax.axhline(RECOGNISED, color="grey", lw=0.8, ls="--")
    ax.axvline(LAYER, color="grey", lw=0.8); ax.set_title(title); ax.set_xlabel("layer")
axes[0].set_ylabel("AUROC on the English refusal direction"); axes[1].legend()
plt.tight_layout()
out = RUNS / "mechanism"; out.mkdir(exist_ok=True)
plt.savefig(out / "separation_by_layer.png", dpi=150)
plt.show()


In [ ]:
import numpy as np
np.savez_compressed(out / "english_refusal_directions.npz",
                    **{str(k): v for k, v in directions.items()})
(out / "tables.json").write_text(json.dumps({"model": MODEL, "layer": LAYER, "threshold": RECOGNISED,
    "direction_data": "BeaverTails 30k_test harmful (n=128) vs Alpaca no-input (n=128), seed 0",
    "tables": tables}, indent=1), encoding="utf-8")
for f in out.iterdir():
    api.upload_file(path_or_fileobj=str(f), path_in_repo=f"mechanism/{f.name}",
                    repo_id=REPO, repo_type="dataset")
print("uploaded", [f.name for f in out.iterdir()], "->", REPO)
